# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HSB-25/flyrank-ml-internship-W1-Run-the-Starter-Notebooks/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task type: Classification (binary) — with the output used as a scoring/ranking signal.

The underlying question is "will this page's traffic trend flip to down?" — a yes/no
question with an observable label (trend_direction == 'down'), which is a classification
setup. But the action a content strategist takes isn't "act on every flagged page equally" —
it's "work down a prioritized queue." So the model's probability output (not just the 0/1
class) becomes a priority score, and pages get ranked by that score. Classification
underneath, ranking/scoring on top, for the action it actually supports.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Label values:", df["is_declining_label"].unique())
print(df["is_declining_label"].value_counts(normalize=True).round(3))


Label values: [1 0]
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: is_declining_label, defined as trend_direction == "down" (down = last-30-day
impressions at least 20% below the previous 30 days).

Where it comes from: it is derived from an observed metric (trend_pct, the real percent
change in impressions) but defined by a rule — a fixed -20% cutoff, not something a human
labeled by outcome after the fact. The underlying signal (trend_pct) is a real, measured
outcome, but the yes/no cutoff is a chosen threshold, so I'll treat the model's output as
directional risk, not ground truth. Also: trend_direction and trend_pct are the label's
source, so they can never appear as input features — that would just teach the model to
reconstruct its own label.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(df[["content_id", "trend_pct", "trend_direction", "is_declining_label"]].head(8))

check = df[df["trend_direction"] == "down"]["trend_pct"].max()
print("\nMax trend_pct among rows labeled 'down':", check, "(should be < -20)")


             content_id  trend_pct trend_direction  is_declining_label
0  content_304f48230142      -41.4            down                   1
1  content_a1fb4e703a9e      -57.7            down                   1
2  content_9aa793d4d895      -60.9            down                   1
3  content_331d6c4de07b      -13.8          stable                   0
4  content_d99b7a2d90ca      -34.7            down                   1
5  content_d4084a4bc775      -38.9            down                   1
6  content_9a34b442b552      -92.3            down                   1
7  content_a63219c6e95a        0.6          stable                   0

Max trend_pct among rows labeled 'down': -20.0 (should be < -20)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Metric: precision@50 (with ROC-AUC as a supporting number during model development).

A content team doesn't review all 30,000 pages — they work through the top of a queue in a
sprint, say the top 50 pages. So the metric that matters is: of the top 50 pages the model
ranks highest-risk, how many are actually declining? That's precision@K, not overall accuracy
— accuracy would be dominated by the ~54% base rate and wouldn't tell a strategist anything
about queue quality.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
base_rate = df["is_declining_label"].mean()
print(f"Base decline rate: {base_rate:.1%}")
print(f"A random/naive queue of 50 pages would contain ~{base_rate*50:.0f} true decliners by chance.")
print("Success = a model-ranked top-50 queue that clears this bar by a wide margin (target: precision@50 > 0.70).")


Base decline rate: 54.2%
A random/naive queue of 50 pages would contain ~27 true decliners by chance.
Success = a model-ranked top-50 queue that clears this bar by a wide margin (target: precision@50 > 0.70).


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: one row = one content item (one page), not one client and not one day.
The 90-day metrics are already aggregated to the page level, so each row is a single page's
trailing-90-day snapshot for one of 32 clients.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
unit_cols = [
    "content_id", "client_id", "content_type", "content_age_days",
    "avg_position", "ctr", "engagement_rate", "trend_pct", "trend_direction", "is_declining_label"
]
print("One row =", "one content item (page), scored over its trailing 90 days")
df[unit_cols].head(10)

One row = one content item (page), scored over its trailing 90 days


,content_id,client_id,content_type,content_age_days,avg_position,ctr,engagement_rate,trend_pct,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,187,10.6,0.76,5.88,-41.4,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,445,20.3,0.05,0.00,-57.7,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,141,36.5,0.09,0.00,-60.9,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,463,6.2,0.49,1.28,-13.8,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,263,44.0,0.13,0.00,-34.7,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,147,8.5,0.03,0.00,-38.9,down,1
6,content_9a34b442b552,client_8722616204,keyword article,90,7.0,0.00,0.00,-92.3,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,445,21.2,0.06,3.57,0.6,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,90,46.0,0.09,5.88,-58.8,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,257,4.9,0.16,0.00,-29.2,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule like "flag anything older than 6 months" or "flag anything below position 20"
fails because decline isn't driven by one signal in isolation — it's a combination of age,
position, CTR, and engagement that interacts differently per content type and per client.
Hand-writing every combination of thresholds across 44 columns and 32 clients isn't
realistic, and any single-column rule will flag far too many false positives to be a usable
priority queue.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
sample = df[df["position_tier"] == "page_1"][
    ["content_id", "avg_position", "ctr", "engagement_rate", "content_age_days", "is_declining_label"]
]

print("Same position tier ('page_1'), opposite outcomes:")
print(sample.groupby("is_declining_label")[["ctr", "engagement_rate", "content_age_days"]].mean().round(2))
print("\n-> A rule keyed only on position_tier can't separate these two groups; ctr/engagement/age diverge together.")

Same position tier ('page_1'), opposite outcomes:
                     ctr  engagement_rate  content_age_days
is_declining_label                                         
0                   0.94             3.15            257.35
1                   0.43             2.78            231.77

-> A rule keyed only on position_tier can't separate these two groups; ctr/engagement/age diverge together.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.